Object attribute access

Access to object attributes can be controlled via the following magic methods

Expression example	Magic method	Implementation meaning or purpose

- object.attribute
  - \__getattr__(self, attribute)
  - responsible for handling access to a non-existing attribute
- object.attribute
  - \__getattribute__(self, attribute)
  - responsible for handling access to an existing attribute
- object.attribute = value
    - \__setattr__(self, attribute, value)
    - responsible for setting an attribute value
- del object.attribute
    - \__delattr__(self, attribute)	responsible for deleting an attribute

**Key points about `__getattr__`:**

1. **Only called for missing attributes** - It's not called for attributes that actually exist on the object
2. **Fallback mechanism** - Provides a way to handle undefined attribute access gracefully
3. **Must return a value** - If you don't return anything, `None` is returned by default
4. **Can raise AttributeError** - If you want to maintain normal "attribute not found" behavior

**Common use cases:**
- Creating proxy objects
- Building flexible APIs
- Implementing attribute delegation
- Creating mock objects for testing

**Suggested Python packages** that commonly use this pattern:
- `unittest.mock` - Uses `__getattr__` in Mock objects
- `requests` - Uses it for dynamic API endpoints
- `pandas` - Uses it for dynamic column access in DataFrames

In [4]:
class Person:
    def __init__(self):
        self.name = "Alice"

    def __getattribute__(self, attr):
        print("Inside getattribute")
        return object.__getattribute__(self, attr)

    def __getattr__(self, attr):
        print("Inside getattr")
        return "Unknown"
p = Person()
print(p.name)
print(p.age)

Inside getattribute
Alice
Inside getattribute
Inside getattr
Unknown


In [1]:
class DynamicAttributes:
    def __init__(self):
        self.existing_attr = "I exist!"
    
    def __getattr__(self, name):
        # This is called only when the attribute doesn't exist
        return f"Dynamic attribute '{name}' was accessed!"

# Example usage
obj = DynamicAttributes()

# Accessing existing attribute (normal behavior)
print(obj.existing_attr)  # Output: "I exist!"

# Accessing non-existing attributes (triggers __getattr__)
print(obj.some_random_attr)  # Output: "Dynamic attribute 'some_random_attr' was accessed!"
print(obj.another_attr)      # Output: "Dynamic attribute 'another_attr' was accessed!"
print(obj.__dict__)

I exist!
Dynamic attribute 'some_random_attr' was accessed!
Dynamic attribute 'another_attr' was accessed!
{'existing_attr': 'I exist!'}


**Key differences between `__getattribute__` and `__getattr__`:**

| `__getattribute__` | `__getattr__` |
|-------------------|---------------|
| Called for **ALL** attribute access | Called only for **missing** attributes |
| Always executed first | Only executed as fallback |
| Higher risk of infinite recursion | Safer to implement |
| Must use `object.__getattribute__()` to get actual values | Can access attributes normally |

**Important considerations:**
1. **Recursion risk** - You must use `object.__getattribute__(self, name)` to avoid infinite loops
2. **Performance impact** - Called for every attribute access, so can slow down your code
3. **Debugging complexity** - Can make debugging more difficult since it intercepts all access

**Common use cases:**
- Logging/monitoring attribute access
- Implementing security/access control
- Creating debugging tools
- Building sophisticated proxy objects

**Suggested Python packages** that might use similar patterns:
- `sqlalchemy` - For ORM attribute access tracking
- `django` - For model field access
- `pytest` - For test object introspection
- `wrapt` - For creating decorators and proxies

In [1]:
class AttributeLogger:
    def __init__(self):
        # Use object.__setattr__ to avoid infinite recursion
        object.__setattr__(self, 'name', 'Alice')
        object.__setattr__(self, 'age', 30)
        object.__setattr__(self, '_access_log', [])
    
    def __getattribute__(self, name):
        # This is called for EVERY attribute access
        print(f"Accessing attribute: {name}")
        
        # Get the actual value using object.__getattribute__ to avoid recursion
        value = object.__getattribute__(self, name)
        
        # Log the access (be careful with recursion here)
        if name != '_access_log':
            access_log = object.__getattribute__(self, '_access_log')
            access_log.append(name)
        
        return value

# Example usage
person = AttributeLogger()

# Every attribute access triggers __getattribute__
print(person.name)        # Prints: "Accessing attribute: name" then "Alice"
print(person.age)         # Prints: "Accessing attribute: age" then "30"
print(person._access_log) # Shows: ['name', 'age']
print(person.xxx) # not an existing attribute

Accessing attribute: name
Alice
Accessing attribute: age
30
Accessing attribute: _access_log
['name', 'age']
Accessing attribute: xxx


AttributeError: 'AttributeLogger' object has no attribute 'xxx'

**Key points about `__delattr__`:**

1. **Called by `del` statement** - Triggered when you use `del obj.attribute`
2. **Can prevent deletion** - Raise `AttributeError` to block deletion
3. **Must call `object.__delattr__()`** - To actually perform the deletion
4. **Useful for validation** - Can implement business logic around attribute removal

**Common use cases:**
- **Access control** - Preventing deletion of critical attributes
- **Logging/auditing** - Tracking when attributes are removed
- **Data validation** - Ensuring object remains in valid state
- **Cleanup operations** - Performing additional cleanup when attributes are deleted

**Alternative approaches without `__delattr__`:**
- Use properties with custom deleters
- Implement explicit removal methods
- Use `__slots__` to restrict attribute creation/deletion

**Suggested Python packages** that might use similar patterns:
- `sqlalchemy` - For ORM field deletion control
- `django` - For model field management
- `attrs`/`dataclasses` - For structured data with deletion control
- `pydantic` - For data validation and field management

In [ ]:
class ProtectedAttributes:
    def __init__(self):
        self.name = "Alice"
        self.age = 30
        self.email = "alice@example.com"
        self._protected = "secret data"
        self._deletion_log = []
    
    def __delattr__(self, name):
        # This is called when using 'del obj.attribute'
        print(f"Attempting to delete attribute: {name}")
        
        # Protect certain attributes from deletion
        if name.startswith('_'):
            raise AttributeError(f"Cannot delete protected attribute '{name}'")
        
        if name == 'name':
            raise AttributeError("Name cannot be deleted!")
        
        # Log the deletion
        self._deletion_log.append(name)
        
        # Actually delete the attribute
        object.__delattr__(self, name)
        print(f"Successfully deleted: {name}")

# Example usage
person = ProtectedAttributes()

print(f"Initial attributes: {vars(person)}")

# Successful deletion
del person.email  # Prints deletion messages, removes email

# Failed deletions (will raise AttributeError)
try:
    del person.name  # Protected - will fail
except AttributeError as e:
    print(f"Error: {e}")

try:
    del person._protected  # Protected attribute - will fail
except AttributeError as e:
    print(f"Error: {e}")

print(f"Final attributes: {vars(person)}")
print(f"Deletion log: {person._deletion_log}")